In [1]:
import os
from pathlib import Path

import geopandas as gpd
import numpy as np

In [2]:
data_path = Path(os.environ["DATA_PATH"])
population_grids_path = Path(os.environ["GRID_PATH"])

In [3]:
df_agebs = gpd.read_file(
    population_grids_path / "final" / "zone_agebs" / "shaped" / "2020" / "02.2.03.gpkg",
).to_crs("EPSG:6372")

In [10]:
gpd.read_file(data_path / "esp_pub")["TIPO"].value_counts()

TIPO
ESPACIO DEPORTIVO         273
JARDIN VECINAL            247
TRIANGULOS                 57
CAMELLONES                 46
GLORIETAS                  23
JARDINES                   18
PARQUE DE BARRIO            6
CUCHILLAS                   5
PLAZA CIVICA                2
JARDINES (VIVERO MPAL)      1
PARQUE URBANO               1
Name: count, dtype: int64

In [ ]:
wanted_types = [
    "JARDIN VECINAL",
    "JARDINES",
    "PARQUE DE BARRIO",
    "JARDINES (VIVERO MPAL)",
    "PARQUE URBANO",
    "ESPACIO DEPORTIVO",
    "TRIANGULOS",
    "CAMELLONES",
    "GLORIETAS",
]

df_parks = (
    gpd.read_file(data_path / "esp_pub")
    .query("TIPO.isin(@wanted_types)")
    .to_crs("EPSG:6372")
    .assign(geometry=lambda df: df["geometry"].force_2d())
)

df_parks_circle = df_parks.assign(
    radius=lambda df: df["Sup_M2"].divide(np.pi).pow(0.5),
    geometry=lambda df: df["geometry"].buffer(df["radius"]),
)

df_parks.to_file("./parks.gpkg")
df_parks_circle.to_file("./parks_circle.gpkg")

In [12]:
df_parks["Sup_M2"].sum() / df_agebs["POBTOT"].sum()

2.9847829567604367

In [6]:
df_parks["TIPO"].value_counts()

TIPO
ESPACIO DEPORTIVO         273
JARDIN VECINAL            247
JARDINES                   18
PARQUE DE BARRIO            6
JARDINES (VIVERO MPAL)      1
PARQUE URBANO               1
Name: count, dtype: int64